In [1]:
print("helo")

helo


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, hilbert, spectrogram
from scipy.fft import fft, fftfreq

In [4]:
BEARING_PARAMS = {
    "BPFI": 5.415,   # Ball Pass Frequency Inner race
    "BPFO": 3.585,   # Ball Pass Frequency Outer race
    "BSF":  2.357,   # Ball Spin Frequency
    "FTF":  0.3983,  # Fundamental Train Frequency (cage)
}
 
# Load speed → shaft frequency map (RPM → Hz)
LOAD_SPEED_MAP = {
    0: 1797 / 60,   # 0 HP  → ~29.95 Hz
    1: 1772 / 60,   # 1 HP  → ~29.53 Hz
    2: 1750 / 60,   # 2 HP  → ~29.17 Hz
    3: 1730 / 60,   # 3 HP  → ~28.83 Hz
}
 
FS = 12_000   # Drive-end sampling rate used in CWRU dataset (Hz)

In [5]:
def load_cwru_mat(filepath: str):
   
    mat = loadmat(filepath, squeeze_me=True)
    # Find the drive-end accelerometer key
    de_key = next(
        (k for k in mat if "DE" in k.upper() and "time" in k.lower()), None
    )
    if de_key is None:
        raise KeyError(
            f"Could not find a Drive-End key in {filepath}.\n"
            f"Available keys: {[k for k in mat if not k.startswith('_')]}"
        )
    signal = mat[de_key].astype(np.float64)
    return signal

In [6]:
def compute_fault_freqs(shaft_hz: float):
    return {name: mult * shaft_hz for name, mult in BEARING_PARAMS.items()}

In [7]:
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype="band")
    return b, a

In [ ]:
def envelope_spectrum(signal, fs, lowcut=2_000, highcut=5_000):

    b, a = butter_bandpass(lowcut, highcut, fs)
    filtered = filtfilt(b, a, signal)
    analytic = hilbert(filtered)
    envelope = np.abs(analytic) - np.abs(analytic).mean()
    N = len(envelope)
    freqs = fftfreq(N, 1 / fs)[: N // 2]
    amp = (2 / N) * np.abs(fft(envelope))[: N // 2]
    return freqs, amp

In [14]:
def analyze_bearing(filepath: str, load: int = 0, n_harmonics: int = 5,segment_sec: float = 0.5):
    signal = load_cwru_mat(filepath)
    shaft_hz = LOAD_SPEED_MAP[load]
    fault_freqs = compute_fault_freqs(shaft_hz)
 
    label = filepath.split("/")[-1].replace(".mat", "")
    N = len(signal)
    t = np.arange(N) / FS
 
    freqs_full = fftfreq(N, 1 / FS)[: N // 2]
    fft_amp = (2 / N) * np.abs(fft(signal))[: N // 2]
 
    env_freqs, env_amp = envelope_spectrum(signal, FS)
 
    f_spec, t_spec, Sxx = spectrogram(signal, fs=FS, nperseg=512, noverlap=256)
 
    stats = {
        "RMS":      np.sqrt(np.mean(signal ** 2)),
        "Peak":     np.max(np.abs(signal)),
        "Crest":    np.max(np.abs(signal)) / np.sqrt(np.mean(signal ** 2)),
        "Kurtosis": float(np.mean((signal - signal.mean()) ** 4) /
                          np.mean((signal - signal.mean()) ** 2) ** 2),
        "Skewness": float(np.mean((signal - signal.mean()) ** 3) /
                          np.mean((signal - signal.mean()) ** 2) ** 1.5),
        
    }
    fig = plt.figure(figsize=(20, 22), facecolor="#0f0f1a")
    fig.suptitle(
        f"CWRU Bearing Analysis  ·  {label}  ·  Load {load} HP  ·  "
        f"Shaft ≈ {shaft_hz*60:.0f} RPM",
        fontsize=16, color="white", fontweight="bold", y=0.98,
    )
 
DARK_BG  = "#0f0f1a"
PANEL_BG = "#1a1a2e"
GRID_CLR = "#2a2a4a"
TEXT_CLR = "#e0e0f0"
 
FAULT_COLORS = {
        "BPFI": "#ff6b6b",
        "BPFO": "#ffd93d",
        "BSF":  "#6bcb77",
        "FTF":  "#4d96ff",
    }
HARMONIC_ALPHA = [1.0, 0.7, 0.5, 0.35, 0.25]
 
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35,
                           top=0.94, bottom=0.06, left=0.07, right=0.96)
 
def style_ax(ax, title, xlabel, ylabel):
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=TEXT_CLR, labelsize=9)
        ax.xaxis.label.set_color(TEXT_CLR)
        ax.yaxis.label.set_color(TEXT_CLR)
        ax.title.set_color(TEXT_CLR)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID_CLR)
        ax.grid(True, color=GRID_CLR, linewidth=0.5, alpha=0.6)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)


NameError: name 'label' is not defined

<Figure size 2000x2200 with 0 Axes>